# Hackalysis

This notebook contains detailed analysis of the hackathon repositories, including metadata and concept extraction. The goal is to understand the lifecycle of project contributions in relation to hackathon events. Also, to be able to detect Hackathon repositories through various signals

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

from dotenv import load_dotenv
import pandas as pd

load_dotenv()

DATA_ROOT = Path(os.getenv('DATA_ROOT'))
PROVIDER_PREFIX = 'lauzhack'

## Project Based Analysis

### Create Project Based Analysis Data set 

In this data set there is 1 row per project and repo meta data is appended as a list of dicts. This allows us to do project based analysis and also to have all the repos of a project together.

In [108]:
def _load_hackathon_metadata_row(metadata_json_path: Path, year: int) -> dict:
    metadata = json.loads(metadata_json_path.read_text(encoding='utf-8'))
    if isinstance(metadata, list):
        metadata = metadata[0] if metadata else {}
    if not isinstance(metadata, dict):
        metadata = {}

    row = {f'hackathon_{k}': v for k, v in metadata.items()}
    row['hackathon_year'] = year
    return row


def load_projects_analysis_ready(data_root: Path, provider_prefix: str = 'lauzhack') -> pd.DataFrame:
    frames: list[pd.DataFrame] = []

    for folder in sorted(data_root.glob(f'{provider_prefix}-*')):
        if not folder.is_dir():
            continue

        year_str = folder.name.split('-')[-1]
        if not year_str.isdigit():
            continue
        year = int(year_str)

        projects_path = folder / f'{provider_prefix}_projects.parquet'
        github_projects_path = folder / f'{provider_prefix}_github_project_metadata.parquet'
        metadata_json_path = folder / f'{provider_prefix}_metadata.json'

        if not projects_path.exists() or not metadata_json_path.exists():
            continue

        # Prefer project-level GitHub-enriched file when present.
        source_path = github_projects_path if github_projects_path.exists() else projects_path
        df = pd.read_parquet(source_path).copy()
        df['year'] = year

        metadata_row = _load_hackathon_metadata_row(metadata_json_path, year)
        for col, val in metadata_row.items():
            if isinstance(val, (list, dict, tuple, set)):
                df[col] = [val for _ in range(len(df))]
            else:
                df[col] = val

        frames.append(df)

    if not frames:
        return pd.DataFrame()

    merged = pd.concat(frames, ignore_index=True)

    # Harmonize expected GitHub columns even when some years have no github_project_metadata file.
    if 'github_repo_urls' not in merged.columns:
        merged['github_repo_urls'] = [[] for _ in range(len(merged))]
    if 'github_repos_metadata' not in merged.columns:
        merged['github_repos_metadata'] = [[] for _ in range(len(merged))]
    if 'github_repo_count' not in merged.columns:
        merged['github_repo_count'] = 0

    merged['github_repo_count'] = pd.to_numeric(merged['github_repo_count'], errors='coerce').fillna(0).astype(int)

    # Global key for cross-year/provider uniqueness.
    if 'project_uid' in merged.columns:
        merged['global_project_uid'] = merged.apply(
            lambda r: f"{provider_prefix}:{int(r['year'])}:{r['project_uid']}" if pd.notna(r.get('project_uid')) else f"{provider_prefix}:{int(r['year'])}:row:{r.name}",
            axis=1,
        )
    else:
        merged['global_project_uid'] = merged.apply(
            lambda r: f"{provider_prefix}:{int(r['year'])}:row:{r.name}", axis=1
        )

    return merged

#### Exploratory Data Analysis

In [119]:
projects_df = load_projects_analysis_ready(DATA_ROOT, PROVIDER_PREFIX)
print(f"Loaded {len(projects_df)} projects across all years.\n rows = {len(projects_df)} \n columns = {len(projects_df.columns)}")
projects_df.columns


def _canonicalize(v):
    if v is None:
        return ""

    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass

    if isinstance(v, str):
        s = v.strip()
        if not s:
            return ""
        try:
            v = json.loads(s)
        except Exception:
            return s

    if isinstance(v, dict):
        return json.dumps(v, ensure_ascii=False, sort_keys=True)

    if isinstance(v, (list, tuple, set)):
        items = [str(x).strip() for x in v if x is not None and str(x).strip()]
        items = sorted(items)
        return json.dumps(items, ensure_ascii=False)

    return str(v).strip()



projects_df["awards"] = projects_df["awards"].map(_canonicalize)
projects_df["categories"] = projects_df["categories"].map(_canonicalize)
projects_df["team"] = projects_df["team"].map(_canonicalize)
projects_df["github_repo_urls"] = projects_df["github_repo_urls"].map(_canonicalize)


projects_df["title"] = projects_df["title"].map(_canonicalize)
projects_df["project_title"] = projects_df["project_title"].map(_canonicalize)

to_be_dropped_col = []

if (projects_df["awards"].equals(projects_df["categories"])):
    to_be_dropped_col.append("categories")
if projects_df["title"].equals(projects_df["project_title"]):
    to_be_dropped_col.append("project_title")
if projects_df['hackathon_year'].equals(projects_df['year']):
    to_be_dropped_col.append('hackathon_year')

non_analysis_cols = ['hackathon_name', 'hackathon_location', 'hackathon_source_url', 'hackathon_description', 'hackathon_date', 'hackathon_social_links', 'hackathon_extracted_at'] # I am dropping these columns because for now we only focus on lauzhack so year column is enough to ideantify the hackathon
to_be_dropped_col.extend(non_analysis_cols)
to_be_dropped_col.extend(['project_uid', 'id', 'project_id','image_url', 'tags']) # I am dropping this column because it is just a unique identifier and does not have any meaning for analysis

projects_df.drop(columns=to_be_dropped_col, inplace=True)

print(f"After dropping columns: \n columns = {len(projects_df.columns)} \n sample columns = {projects_df.columns.tolist()}")

# Convert hackathon_date to datetime

projects_df['hackathon_date_start'] = pd.to_datetime(projects_df['hackathon_date_start'], errors='coerce')
projects_df['hackathon_date_end'] = pd.to_datetime(projects_df['hackathon_date_end'], errors='coerce')

Loaded 213 projects across all years.
 rows = 213 
 columns = 27
After dropping columns: 
 columns = 12 
 sample columns = ['title', 'description', 'url', 'team', 'awards', 'github_repo_urls', 'github_repo_count', 'github_repos_metadata', 'year', 'hackathon_date_start', 'hackathon_date_end', 'global_project_uid']


In [120]:
projects_df.head()

,title,description,url,team,awards,github_repo_urls,github_repo_count,github_repos_metadata,year,hackathon_date_start,hackathon_date_end,global_project_uid
0,"SightSync, a virtual assistant for visual impa...",An app that describes the surroundings through...,https://github.com/sightsync,"[""Ferran Aran"", ""Oriol Agost Batalla""]","[""1st place overall""]","[""https://github.com/sightsync/.github"", ""http...",4,"[{""input_url"": ""https://github.com/sightsync/....",2023,2023-12-02,2023-12-03,lauzhack:2023:id:1
1,VirtuWheel,Real city driving simulator with hand pose rec...,https://github.com/alvaro-budria/VirtuWheel,"[""Jaume Ros Alonso""]","[""2nd place overall""]","[""https://github.com/alvaro-budria/VirtuWheel""]",1,"[{""input_url"": ""https://github.com/alvaro-budr...",2023,2023-12-02,2023-12-03,lauzhack:2023:id:2
2,It’s not about winning,"""It’s not about winning"" is a cutting-edge app...",https://fastmaildassdas.retool.com/apps/92b217...,"[""Gero Embser"", ""Indira Fömmel"", ""Leonard Eyer""]","[""3rd place overall"", ""AXA challenge winner""]",[],0,[],2023,2023-12-02,2023-12-03,lauzhack:2023:id:3
3,BMS detction HCM,model learning and qualtitative feedback diagn...,https://github.com/Gabriel29062001/hackathon,"[""Eddy BESSAH"", ""Grégoire Longechamp"", ""gabrie...","[""BMS challenge winner"", ""Organizers' prize""]","[""https://github.com/Gabriel29062001/hackathon""]",1,"[{""input_url"": ""https://github.com/Gabriel2906...",2023,2023-12-02,2023-12-03,lauzhack:2023:id:4
4,AWS Challenge,Our take on obtaining useful and compact infor...,https://github.com/mgil4/AWS_LAUZ,"[""Gustavo Vergara Gamboa"", ""Lola Monroy Mir"", ...","[""AWS challenge winner""]","[""https://github.com/mgil4/AWS_LAUZ""]",1,"[{""input_url"": ""https://github.com/mgil4/AWS_L...",2023,2023-12-02,2023-12-03,lauzhack:2023:id:5


In [121]:
projects_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 213 entries, 0 to 212
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   title                  213 non-null    str           
 1   description            213 non-null    str           
 2   url                    213 non-null    str           
 3   team                   213 non-null    str           
 4   awards                 213 non-null    str           
 5   github_repo_urls       213 non-null    str           
 6   github_repo_count      213 non-null    int64         
 7   github_repos_metadata  213 non-null    str           
 8   year                   213 non-null    int64         
 9   hackathon_date_start   213 non-null    datetime64[us]
 10  hackathon_date_end     213 non-null    datetime64[us]
 11  global_project_uid     213 non-null    str           
dtypes: datetime64[us](2), int64(2), str(8)
memory usage: 1.2 MB


In [42]:
# projects_df[['title', 'description', 'team', 'awards', 'hackathon_year']]

<StringDtype(na_value=nan)>

In [93]:
projects_df['hackathon_date']

0        December 2-3
1        December 2-3
2        December 2-3
3        December 2-3
4        December 2-3
            ...      
208    November 22-23
209    November 22-23
210    November 22-23
211    November 22-23
212    November 22-23
Name: hackathon_date, Length: 213, dtype: str

In [13]:
projects_df.info()

## Remove Duplicates
projects_df[projects_df.duplicated(subset=['title'], keep=False)].sort_values('title')[['year', 'title', 'github_repo_urls', 'github_repo_count', 'description', 'team', 'github_repos_metadata']]


<class 'pandas.DataFrame'>
RangeIndex: 213 entries, 0 to 212
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   id                      213 non-null    int64 
 1   title                   213 non-null    str   
 2   description             213 non-null    str   
 3   url                     213 non-null    str   
 4   team                    213 non-null    str   
 5   awards                  97 non-null     str   
 6   categories              97 non-null     str   
 7   hackathon_name          213 non-null    str   
 8   hackathon_year          213 non-null    int64 
 9   hackathon_location      213 non-null    str   
 10  project_uid             213 non-null    str   
 11  project_id              213 non-null    str   
 12  project_title           213 non-null    str   
 13  github_repo_urls        213 non-null    str   
 14  github_repo_count       213 non-null    int64 
 15  github_repos_meta

,year,title,github_repo_urls,github_repo_count,description,team,github_repos_metadata
16,2023,Amazon Review Tools,"[""https://github.com/AliEmreSenel/LauzHack2023""]",1,Our project aims to provide an accessible inte...,"[""Ali Emre Senel"", ""Lorenzo Calda""]","[{""input_url"": ""https://github.com/AliEmreSene..."
17,2023,Amazon Review Tools,"[""https://github.com/AliEmreSenel/LauzHack2023""]",1,Our project aims to provide an accessible inte...,"[""Alberto Paolo Lolli""]","[{""input_url"": ""https://github.com/AliEmreSene..."
18,2023,Bioicons PDB2Vector,"[""https://github.com/bioicons/pdb2vector""]",1,A service to create vector illustrations from ...,"[""Simon Dürr""]","[{""input_url"": ""https://github.com/bioicons/pd..."
44,2023,Bioicons PDB2Vector,"[""https://github.com/bioicons/pdb2vector""]",1,A service to create vector illustrations from ...,"[""Ryoma Maeda"", ""Felix Richter"", ""Laurenz Rasc...","[{""input_url"": ""https://github.com/bioicons/pd..."
31,2023,Legacy LM,[],0,"A personalized conversational AI, preserving v...","[""Alejandro Hernández Cano"", ""Arvind Menon"", ""...",[]
51,2023,Legacy LM,"[""https://github.com/lars-quaedvlieg/Lauzhack-...",1,"A personalized conversational AI model, preser...","[""Somesh Mehra""]","[{""input_url"": ""https://github.com/lars-quaedv..."
35,2023,OpenLogs Lauzhack,"[""https://github.com/EncryptEx/LauzHack23""]",1,Tired of analizing log data by yourself? Try o...,"[""Jaume López Molina""]","[{""input_url"": ""https://github.com/EncryptEx/L..."
59,2023,OpenLogs Lauzhack,"[""https://github.com/EncryptEx/LauzHack23""]",1,Tired of analizing log data by yourself? Try o...,"[""Joffre Alcivar Riera"", ""Pau Carulla Lechosa""]","[{""input_url"": ""https://github.com/EncryptEx/L..."
